Rúbrica: 

A continuación se muestra la rúbrica con la que se va a corregir el examen:


| Apartado/Criterio | Ponderación | Nota |
| :-- | --- | --- |
| Ej. 1.1. Ha tratado de manera adecuada los datos de las columnas. | 1 | 0,5 |
| Ej. 1.1. Ha seguido un criterio adecuado para elegir las entradas del problema. | 1 | 0,75 |
| Ej. 1.2. Ha diseñado bien la red y el sistema de entrenamiento | 1 | 0,75 |
| Ej. 1.2. Ha dimensionado bien la red neuronal. | 1 | 0,75 |
| Ej. 1.2. Ha hecho modificaciones coherentes para conseguir un mejor resultado. | 0,5 | 0 |
| Ej. 1.2. Ha usado su experiencia para valorar si el resultado es válido o no. | 1 | 0 |
| Ej. 2.1. Ha cargado adecuadamente los datos. | 0,5 | 0,5 |
| Ej. 2.2. Ha diseñado bien la red y el sistema de entrenamiento | 1 | 0 |
| Ej. 2.2. Ha dimensionado bien la red neuronal. | 1 | 0 |
| Ej. 2.2. Ha hecho modificaciones coherentes para conseguir un mejor resultado. | 1 | 0 |
| Ej. 2.2. Ha usado su experiencia para valorar si el resultado es válido o no. | 1 | 0 |

TOTAL: 3,25



In [1]:
from os import listdir
from numpy import asarray
from numpy import save
import tensorflow as tf
from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from tensorflow import keras
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVR
from sklearn.metrics import classification_report
import datetime
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

I0000 00:00:1775758015.556305   43047 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1775758015.594403   43047 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775758016.329536   43047 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Ejercicio 1

In [2]:
vuelos = pd.read_csv("vuelos_pakistan.csv")

vuelos.head()

,Flight_ID,Date,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,...,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Delay_Category,On_Time_Status,Weather_Condition,Fuel_Consumption,CO2_Emissions,Customer_Rating,Customer_Feedback
0,PK2026_0001,2026-06-09,June,Tuesday,Jeddah,Islamabad,International,Airbus A320,83.0,120,...,66.67,1140.0,220,Severe,Delayed,Clear,6265l,15662.5kg,4.1,Dreadful customer support
1,PK2026_0002,2026-08-12,August,Wednesday,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,...,99.44,773.0,27,Minor,Delayed,NaN,3516l,8790.0kg,3.6,"Standard flight, nothing special"
2,PK2026_0003,2026-04-20,April,Monday,Doha,Lahore,International,ATR 72,333.0,69,...,98.57,155.0,176,Severe,Delayed,Fog,13538l,33845.0kg,3.0,Tardy arrival but very cozy
3,PK2026_0004,2026-12-07,December,Monday,Jeddah,Lahore,International,Boeing 777,330.0,291,...,83.14,1237.0,87,Moderate,Delayed,NaN,18850l,47125.0kg,NaN,Mediocre experience overall
4,PK2026_0005,2026-05-04,May,Monday,Lahore,Doha,International,Boeing 737,283.0,159,...,99.38,141.0,82,Moderate,Delayed,NaN,13474l,33685.0kg,3.0,Behind schedule but quite relaxed


In [3]:
vuelos["Departure_City"].unique()

array(['Jeddah', 'Dubai', 'Doha', 'Lahore', 'Islamabad', 'Karachi',
       'Kuala Lumpur', 'London'], dtype=object)

In [4]:
vuelos["Arrival_City"].unique()

array(['Islamabad', 'Kuala Lumpur', 'Lahore', 'Doha', 'Dubai', 'London',
       'Karachi', 'Jeddah'], dtype=object)

In [5]:
vuelos["Route_Type"].unique()

array(['International', 'Domestic'], dtype=object)

In [6]:
vuelos["Aircraft_Type"].unique()

array(['Airbus A320', 'ATR 72', 'Boeing 777', 'Boeing 737'], dtype=object)

In [7]:
vuelos["Delay_Category"].unique()

array(['Severe', 'Minor', 'Moderate', 'No Delay'], dtype=object)

In [8]:
vuelos["On_Time_Status"].unique()

array(['Delayed', 'On Time'], dtype=object)

In [9]:
vuelos["Weather_Condition"].unique()	

array(['Clear', nan, 'Fog', 'Rain', 'Storm'], dtype=object)

In [10]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Flight_ID                800 non-null    object 
 1   Date                     800 non-null    object 
 2   Month                    800 non-null    object 
 3   Day_of_Week              800 non-null    object 
 4   Departure_City           800 non-null    object 
 5   Arrival_City             800 non-null    object 
 6   Route_Type               800 non-null    object 
 7   Aircraft_Type            800 non-null    object 
 8   Flight_Duration_Minutes  725 non-null    float64
 9   Passengers               800 non-null    int64  
 10  Seat_Capacity            800 non-null    int64  
 11  Load_Factor_%            800 non-null    float64
 12  Ticket_Price_USD         786 non-null    float64
 13  Delay_Minutes            800 non-null    int64  
 14  Delay_Category           8

In [11]:
vuelos.isnull().sum()

Flight_ID                    0
Date                         0
Month                        0
Day_of_Week                  0
Departure_City               0
Arrival_City                 0
Route_Type                   0
Aircraft_Type                0
Flight_Duration_Minutes     75
Passengers                   0
Seat_Capacity                0
Load_Factor_%                0
Ticket_Price_USD            14
Delay_Minutes                0
Delay_Category               0
On_Time_Status               0
Weather_Condition          279
Fuel_Consumption             0
CO2_Emissions                0
Customer_Rating            127
Customer_Feedback            0
dtype: int64

In [12]:
vuelos = vuelos.drop(columns=["Flight_ID", "Date", "Weather_Condition", "Customer_Rating", "Customer_Feedback"])

In [13]:
vuelos = vuelos.dropna()

In [14]:
vuelos.corr(numeric_only=True)["Ticket_Price_USD"].sort_values(ascending=False)

Ticket_Price_USD           1.000000
Passengers                 0.058469
Seat_Capacity              0.036270
Load_Factor_%              0.002921
Flight_Duration_Minutes    0.001597
Delay_Minutes             -0.015824
Name: Ticket_Price_USD, dtype: float64

In [15]:
vuelos.head()

,Month,Day_of_Week,Departure_City,Arrival_City,Route_Type,Aircraft_Type,Flight_Duration_Minutes,Passengers,Seat_Capacity,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Delay_Category,On_Time_Status,Fuel_Consumption,CO2_Emissions
0,June,Tuesday,Jeddah,Islamabad,International,Airbus A320,83.0,120,180,66.67,1140.0,220,Severe,Delayed,6265l,15662.5kg
1,August,Wednesday,Dubai,Kuala Lumpur,International,Airbus A320,284.0,179,180,99.44,773.0,27,Minor,Delayed,3516l,8790.0kg
2,April,Monday,Doha,Lahore,International,ATR 72,333.0,69,70,98.57,155.0,176,Severe,Delayed,13538l,33845.0kg
3,December,Monday,Jeddah,Lahore,International,Boeing 777,330.0,291,350,83.14,1237.0,87,Moderate,Delayed,18850l,47125.0kg
4,May,Monday,Lahore,Doha,International,Boeing 737,283.0,159,160,99.38,141.0,82,Moderate,Delayed,13474l,33685.0kg


In [16]:
vuelos["Fuel_Consumption"] = vuelos["Fuel_Consumption"].str.replace('l', '').astype(int)
vuelos["CO2_Emissions"] = vuelos["CO2_Emissions"].str.replace('kg', '').astype(float)


In [17]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 713 entries, 0 to 799
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Month                    713 non-null    object 
 1   Day_of_Week              713 non-null    object 
 2   Departure_City           713 non-null    object 
 3   Arrival_City             713 non-null    object 
 4   Route_Type               713 non-null    object 
 5   Aircraft_Type            713 non-null    object 
 6   Flight_Duration_Minutes  713 non-null    float64
 7   Passengers               713 non-null    int64  
 8   Seat_Capacity            713 non-null    int64  
 9   Load_Factor_%            713 non-null    float64
 10  Ticket_Price_USD         713 non-null    float64
 11  Delay_Minutes            713 non-null    int64  
 12  Delay_Category           713 non-null    object 
 13  On_Time_Status           713 non-null    object 
 14  Fuel_Consumption         713 no

In [18]:
vuelos = pd.get_dummies(vuelos, dtype = 'int')

In [19]:
vuelos.shape

(713, 55)

In [20]:
vuelos.corr(numeric_only=True)["Ticket_Price_USD"].sort_values(ascending=False)

Ticket_Price_USD               1.000000
Arrival_City_Jeddah            0.080967
Departure_City_Karachi         0.075604
Passengers                     0.058469
Day_of_Week_Tuesday            0.056840
Month_September                0.056730
Month_March                    0.049383
Month_August                   0.040593
Seat_Capacity                  0.036270
Arrival_City_Karachi           0.034977
Route_Type_Domestic            0.034601
Aircraft_Type_Boeing 777       0.031576
Month_December                 0.029213
Departure_City_Jeddah          0.028708
Delay_Category_Moderate        0.028421
Day_of_Week_Saturday           0.026174
Departure_City_Doha            0.017647
On_Time_Status_On Time         0.017606
Day_of_Week_Wednesday          0.015193
Aircraft_Type_Airbus A320      0.014431
Departure_City_Kuala Lumpur    0.008056
Month_October                  0.007749
Month_July                     0.006792
Load_Factor_%                  0.002921
Day_of_Week_Monday             0.001674


In [21]:
vuelos.info()

<class 'pandas.core.frame.DataFrame'>
Index: 713 entries, 0 to 799
Data columns (total 55 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Flight_Duration_Minutes      713 non-null    float64
 1   Passengers                   713 non-null    int64  
 2   Seat_Capacity                713 non-null    int64  
 3   Load_Factor_%                713 non-null    float64
 4   Ticket_Price_USD             713 non-null    float64
 5   Delay_Minutes                713 non-null    int64  
 6   Fuel_Consumption             713 non-null    int64  
 7   CO2_Emissions                713 non-null    float64
 8   Month_April                  713 non-null    int64  
 9   Month_August                 713 non-null    int64  
 10  Month_December               713 non-null    int64  
 11  Month_February               713 non-null    int64  
 12  Month_January                713 non-null    int64  
 13  Month_July               

In [22]:
vuelos.head()

,Flight_Duration_Minutes,Passengers,Seat_Capacity,Load_Factor_%,Ticket_Price_USD,Delay_Minutes,Fuel_Consumption,CO2_Emissions,Month_April,Month_August,...,Aircraft_Type_ATR 72,Aircraft_Type_Airbus A320,Aircraft_Type_Boeing 737,Aircraft_Type_Boeing 777,Delay_Category_Minor,Delay_Category_Moderate,Delay_Category_No Delay,Delay_Category_Severe,On_Time_Status_Delayed,On_Time_Status_On Time
0,83.0,120,180,66.67,1140.0,220,6265,15662.5,0,0,...,0,1,0,0,0,0,0,1,1,0
1,284.0,179,180,99.44,773.0,27,3516,8790.0,0,1,...,0,1,0,0,1,0,0,0,1,0
2,333.0,69,70,98.57,155.0,176,13538,33845.0,1,0,...,1,0,0,0,0,0,0,1,1,0
3,330.0,291,350,83.14,1237.0,87,18850,47125.0,0,0,...,0,0,0,1,0,1,0,0,1,0
4,283.0,159,160,99.38,141.0,82,13474,33685.0,0,0,...,0,0,1,0,0,1,0,0,1,0


In [ ]:
#CORRECCIÓN: En esta primera parte, deberías haber usado fillna. Al hacer dropna pierdes innecesariamente datos. 
# En cuanto a la elección de columnas, has borrado rating, que es numérica y puede aportar.
X = vuelos.drop('Ticket_Price_USD', axis=1)
y = vuelos['Ticket_Price_USD']


In [24]:

scaler = StandardScaler()
X = scaler.fit_transform(vuelos.drop(columns=['Ticket_Price_USD']))
y = scaler.fit_transform(vuelos[['Ticket_Price_USD']])

X_train_full, X_test, y_train_full, y_test = train_test_split(X,y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full)


In [25]:
'''
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
'''


'\nX_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)\nscaler = StandardScaler()\nX_train = scaler.fit_transform(X_train)\nX_test = scaler.transform(X_test)\n'

In [26]:
X.shape 

(713, 54)

In [27]:
y.shape

(713, 1)

In [28]:
model = keras.Sequential([
    keras.layers.Dense(400, activation="relu", input_shape=X_train.shape[1:]),
    keras.layers.Dense(250, activation="relu"),
    keras.layers.Dense(1)
])

/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
W0000 00:00:1775758019.234995   43047 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [29]:
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01), loss="mean_squared_error", metrics=['r2_score'])

In [ ]:
history = model.fit(X_train, y_train, epochs=20, validation_split=0.1)
#CORRECCIÓN: El diseño está bien. Has dejado entrenar pocas épocas.
#El dimensioanmiento es grande para el número de entradas que tienes.

Epoch 1/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1.0214 - r2_score: -0.0307 - val_loss: 0.9339 - val_r2_score: 0.0235
Epoch 2/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.9461 - r2_score: 0.0453 - val_loss: 0.9811 - val_r2_score: -0.0259
Epoch 3/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.8473 - r2_score: 0.1450 - val_loss: 1.0643 - val_r2_score: -0.1128
Epoch 4/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.7875 - r2_score: 0.2054 - val_loss: 1.3116 - val_r2_score: -0.3715
Epoch 5/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.7403 - r2_score: 0.2530 - val_loss: 0.9843 - val_r2_score: -0.0292
Epoch 6/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6970 - r2_score: 0.2967 - val_loss: 1.0151 - val_r2_score: -0.0615
Epoch 7/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6530 - r2_score: 0.3411 - val_loss: 1.1110 - val_r2_score: -0.1617
Epoch 8/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6062 - r2_score: 0.3883 - val_loss: 1.1313 - val_r

In [ ]:
model.evaluate(X,y)
#CORRECCIÓN: No me has dejado nada más que un intento y no has usado herrameintas para valorar el resultado.

23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7064 - r2_score: 0.2936 


[0.7064250707626343, 0.2935747504234314]

In [ ]:
model.save("carmen_vuelos")

## Ejercicio 2

In [ ]:
folders = listdir('./cartas')

photos = []
labels = []

for idx,folder in enumerate(folders):
    for file in listdir('./cartas/'+folder):
        photo = load_img('./cartas/'+folder+'/' +file, color_mode='grayscale')
        photo = img_to_array(photo)
        photos.append(photo)
        labels.append(float(idx))
        del photo
    print (idx)

#CORRECCIÓN: En cuanto a la carga de datos, yo creo que tendrías que haber reducido un poco el tamaño de las fotos para poder hacer
# el ejercicio.
    

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52


In [33]:
photos = asarray(photos)
labels = asarray(labels)


In [34]:
photos = photos / 255.0
X = photos
X = X.reshape(7624, -3)
y = labels

In [35]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.1, random_state=42)

In [36]:
X.shape

(7624, 50176)

In [37]:
y.shape

(7624,)

In [ ]:
model = keras.Sequential([
    keras.layers.Dense(2000, activation="relu", input_shape=X_train.shape[1:]),
    keras.layers.Dense(1200, activation="relu"),
    keras.layers.Dense(500, activation="relu"),
    keras.layers.Dense(1, activation="softmax")
])
# CORRECCIÓN: El diseño de la red tendría que tener 53 neuronas en la capa de salida, para ajustarse al diseño de multiclass.
# El  dimensionamiento también está mal ya que ni te acercas a ese 2/3 de neuronas en la capa de entrada.
#No me has peusto varios intentos ni tampoco has usado métodos para valorar los resultados.

In [39]:
model.compile(optimizer=keras.optimizers.SGD(learning_rate=0.01), loss="categorical_crossentropy", metrics=['accuracy'])

In [40]:
history = model.fit(X_train, y_train, epochs=10, validation_split=0.1)

Epoch 1/10


/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/ops/nn.py:947: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/home/ciabd01/anaconda3/lib/python3.13/site-packages/keras/src/losses/losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)


193/193 ━━━━━━━━━━━━━━━━━━━━ 23s 120ms/step - accuracy: 0.0175 - loss: nan - val_accuracy: 0.0146 - val_loss: nan
Epoch 2/10
193/193 ━━━━━━━━━━━━━━━━━━━━ 23s 118ms/step - accuracy: 0.0177 - loss: nan - val_accuracy: 0.0146 - val_loss: nan
Epoch 3/10
193/193 ━━━━━━━━━━━━━━━━━━━━ 23s 120ms/step - accuracy: 0.0177 - loss: nan - val_accuracy: 0.0146 - val_loss: nan
Epoch 4/10
193/193 ━━━━━━━━━━━━━━━━━━━━ 23s 117ms/step - accuracy: 0.0177 - loss: nan - val_accuracy: 0.0146 - val_loss: nan
Epoch 5/10
193/193 ━━━━━━━━━━━━━━━━━━━━ 23s 120ms/step - accuracy: 0.0177 - loss: nan - val_accuracy: 0.0146 - val_loss: nan
Epoch 6/10
193/193 ━━━━━━━━━━━━━━━━━━━━ 24s 126ms/step - accuracy: 0.0177 - loss: nan - val_accuracy: 0.0146 - val_loss: nan
Epoch 7/10
193/193 ━━━━━━━━━━━━━━━━━━━━ 39s 116ms/step - accuracy: 0.0177 - loss: nan - val_accuracy: 0.0146 - val_loss: nan
Epoch 8/10
193/193 ━━━━━━━━━━━━━━━━━━━━ 23s 117ms/step - accuracy: 0.0177 - loss: nan - val_accuracy: 0.0146 - val_loss: nan
Epoch 9/10


In [41]:
model.evaluate(X,y)

239/239 ━━━━━━━━━━━━━━━━━━━━ 6s 23ms/step - accuracy: 0.0182 - loss: nan


[nan, 0.018231898546218872]

In [ ]:
model.save("carmen")